# G4-DNABERT · *Octopus bimaculoides* genome-wide G4 prediction

**Model:** Kouzine fine-tuned head on DNABERT-6mer base  
**Genome:** GCF_001194135.2 (ASM119413v2)  

### Before running
1. `Runtime → Change runtime type → T4 GPU`
2. Upload your `Squid_export.zip` when Cell 2 asks for it  
   (or mount Google Drive and set `SQUID_DIR` manually)


In [2]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install -q transformers biopython scipy scikit-learn tqdm pandas

In [3]:
# ── 2. Squid_export.zip уже загружен вручную ──────────────────────────────
import os, zipfile

WORK = '/content/g4_octopus'
os.makedirs(WORK, exist_ok=True)

# Укажите путь к вашему zip (обычно /content/Squid_export.zip)
ZIP_PATH = '/content/Squid_export.zip'

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(WORK)

SQUID_DIR = os.path.join(WORK, 'Squid_export')
print('Extracted to:', SQUID_DIR)
print(os.listdir(SQUID_DIR))

Extracted to: /content/g4_octopus/Squid_export
['dna_tokenizer.py', '__pycache__', 'dnabert_mm_fold_0_kouzine_g4', '6-new-12w-0', '9_Predict_on_fasta-g4.ipynb', '10_Generate_bed_files.ipynb']


In [4]:
# ── 3. Скачать геном прямо в Colab ────────────────────────────────────────
import os

WORK = '/content/g4_octopus'
GENOME_GZ  = os.path.join(WORK, 'GCF_001194135.2_ASM119413v2_genomic.fna.gz')
GENOME_FNA = GENOME_GZ[:-3]

if not os.path.exists(GENOME_FNA):
    !wget -q --show-progress -c \
        -O {GENOME_GZ} \
        "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/001/194/135/GCF_001194135.2_ASM119413v2/GCF_001194135.2_ASM119413v2_genomic.fna.gz"
    !gunzip -k {GENOME_GZ}

print('Sequences:', end=' ')
!grep -c '>' {GENOME_FNA}

Sequences: 145327


In [5]:
# ── 4. Imports & helpers ───────────────────────────────────────────────────
import sys, gc, pickle
import numpy as np
import torch
from Bio import SeqIO
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from transformers import BertConfig, BertForTokenClassification

sys.path.insert(0, SQUID_DIR)
from dna_tokenizer import DNATokenizer, seq2kmer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# ── sequence splitting / stitching ─────────────────────────────────────────
def split_seq(seq, length=512, pad=16):
    res = []
    n = len(seq)
    for st in range(0, n, length - pad):
        if st > 0 and st + pad >= n:
            break
        res.append(seq[st: min(st + length, n)])
    return res

def stitch_np_seq(np_seqs, pad=16):
    total = sum(s.shape[-1] for s in np_seqs) - pad * (len(np_seqs) - 1)
    res = np.empty(total, dtype=np_seqs[0].dtype)
    pos = 0
    for i, s in enumerate(np_seqs):
        if i > 0: pos -= pad
        res[pos: pos + s.shape[-1]] = s[0]
        pos += s.shape[-1]
    return res

class PredDataset(Dataset):
    def __init__(self, seq, tok):
        self.pieces = split_seq(seq2kmer(seq.upper(), 6).split(' '), 512, 16)
        self.tok = tok
    def __len__(self): return len(self.pieces)
    def __getitem__(self, i):
        ids = self.tok.encode_plus(self.pieces[i], add_special_tokens=False,
                                   max_length=512)['input_ids']
        return torch.LongTensor(ids)

# ── NW-buffer helpers ──────────────────────────────────────────────────────
PRED_DIR = os.path.join(WORK, 'predictions_g4')
os.makedirs(PRED_DIR, exist_ok=True)
NW_FILE  = os.path.join(PRED_DIR, 'NW_all.pickle')

try:
    existing_nw = pickle.load(open(NW_FILE, 'rb'))
    if not isinstance(existing_nw, dict): existing_nw = {}
except FileNotFoundError:
    existing_nw = {}

nw_buffer = {}

def already_done(name):
    if name.startswith('NW_'):
        return name in existing_nw or name in nw_buffer
    return os.path.exists(os.path.join(PRED_DIR, f'{name}.pickle'))

def save_pred(name, arr):
    if name.startswith('NW_'):
        nw_buffer[name] = arr
    else:
        pickle.dump(arr, open(os.path.join(PRED_DIR, f'{name}.pickle'), 'wb'))

def flush_nw():
    if nw_buffer:
        merged = {**existing_nw, **nw_buffer}
        pickle.dump(merged, open(NW_FILE, 'wb'))
        print(f'  NW_all.pickle: {len(merged)} scaffolds')

Device: cuda


In [6]:
# ── 5. Load tokenizer & model ──────────────────────────────────────────────
BASE_DIR = os.path.join(SQUID_DIR, '6-new-12w-0')
HEAD_DIR = os.path.join(SQUID_DIR, 'dnabert_mm_fold_0_kouzine_g4')

tokenizer = DNATokenizer.from_pretrained(BASE_DIR)
config    = BertConfig.from_pretrained(os.path.join(BASE_DIR, 'config.json'))
model     = BertForTokenClassification.from_pretrained(HEAD_DIR, config=config)
model.to(device).eval()
print('Model ready on', device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: /content/g4_octopus/Squid_export/dnabert_mm_fold_0_kouzine_g4
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready on cuda


In [ ]:
# ── 6. Genome-wide prediction ──────────────────────────────────────────────
# Estimated time on T4 GPU: ~3–5 h for full genome
# Resume-safe: already-processed sequences are skipped automatically.

BATCH_SIZE = 8  # increase to 16 if GPU RAM allows

try:
    for rec in SeqIO.parse(GENOME_FNA, 'fasta'):
        name, seq = rec.id, str(rec.seq)
        print(f'\n[{name}]  len={len(seq):,}')

        if already_done(name):
            print('  skipping (done)')
            continue

        ds  = PredDataset(seq, tokenizer)
        dl  = DataLoader(ds, batch_size=BATCH_SIZE)
        buf = []

        with torch.no_grad():
            for batch in tqdm(dl, leave=False):
                scores = torch.softmax(model(batch.to(device))['logits'],
                                       dim=-1)[:, :, 1].cpu().numpy()
                buf.append(scores)

        arr = stitch_np_seq(buf)
        save_pred(name, arr)
        print(f'  max={arr.max():.3f}  mean={arr.mean():.4f}')
        del ds, dl, buf, arr; gc.collect(); torch.cuda.empty_cache()

finally:
    flush_nw()

print('\nAll done. Predictions in:', PRED_DIR)


[NC_068981.1]  len=199,874,329


In [ ]:
# ── 7. Predictions → BED (Otsu threshold) ─────────────────────────────────
import scipy.ndimage
import pandas as pd

def otsu_threshold(values, bins=512):
    hist, edges = np.histogram(values, bins=bins, range=(0., 1.))
    centers = (edges[:-1] + edges[1:]) / 2
    prob = hist.astype(float) / hist.sum()
    w0 = np.cumsum(prob)
    w1 = 1 - w0
    mu0 = np.cumsum(prob * centers) / np.maximum(w0, 1e-12)
    mu1 = (np.cumsum((prob * centers)[::-1])[::-1]) / np.maximum(w1, 1e-12)
    return float(centers[np.argmax(w0 * w1 * (mu0 - mu1) ** 2)])

def extract_segments(arr, name, thr, min_len, records):
    labeled, n = scipy.ndimage.label(arr > thr)
    for slc in scipy.ndimage.find_objects(labeled):
        if slc is None: continue
        s, e = slc[0].start, slc[0].stop
        if e - s >= min_len:
            records.append((name, s, e, float(arr[s:e].mean())))

# load all pickles
all_arrs = {}
for fname in os.listdir(PRED_DIR):
    d = pickle.load(open(os.path.join(PRED_DIR, fname), 'rb'))
    if isinstance(d, dict): all_arrs.update(d)
    elif isinstance(d, np.ndarray): all_arrs[fname.replace('.pickle','')] = d

# Otsu on subsampled pool
pool = np.concatenate([a[::max(1, len(a)//200_000)] for a in all_arrs.values()])
thr  = otsu_threshold(pool)
print(f'Otsu threshold: {thr:.4f}')

records = []
for name, arr in sorted(all_arrs.items()):
    extract_segments(arr, name, thr, min_len=4, records=records)

df = pd.DataFrame(records, columns=['chrom','start','end','score'])
df.sort_values(['chrom','start'], inplace=True)
print(f'Peaks: {len(df):,}   Median width: {(df.end-df.start).median():.0f} nt')

BED_OUT = os.path.join(WORK, 'octopus_g4_kouzine.bed')
df.to_csv(BED_OUT, sep='\t', index=False, header=False, float_format='%.4f')
print('BED written:', BED_OUT)

In [ ]:
# ── 8. Download results ────────────────────────────────────────────────────
import shutil
from google.colab import files

# Pack all pickles + BED into one archive
archive = os.path.join(WORK, 'g4_octopus_results')
shutil.make_archive(archive, 'zip', WORK, 'predictions_g4')
shutil.copy(BED_OUT, WORK)

files.download(BED_OUT)
files.download(archive + '.zip')